In [ ]:
import pandas as pd
import numpy as np
import pickle

# 1. Load Data
file_path = '/content/drive/MyDrive/wheat_yield_india/dataset/final_india_wheat_dataset.csv'
df_india = pd.read_csv(file_path)

# 2. Filter for the modern era
df_india = df_india[(df_india['Year'] >= 1995) & (df_india['Year'] <= 2017)].copy()

# 3. FIX: Aggregate duplicate District-Year rows to remove noise
agg_dict = {
    'Yield': 'mean',
    'Yield_Lag1': 'mean',
    'Avg_Season_Tmax': 'mean',
    'Total_Season_Rain': 'mean',
    'Sowing_Rainfall': 'mean',
    'Terminal_Heat_Tmax': 'mean',
    'Temperature_Anomaly': 'mean',
    'Rainfall_Deviation': 'mean'
}
df_india = df_india.groupby(['State', 'Dist Code', 'Year'], as_index=False).agg(agg_dict)

# 4. FIX: Strict Time-Based Splitting (Train: 1995-2013, Test: 2014-2017)
train_data = df_india[df_india['Year'] <= 2013].copy()
test_data = df_india[df_india['Year'] > 2013].copy()

# 5. FIX: Target Encoding for Geography (Instead of Label Encoding)
# We replace the State/District with its historical average yield from the training set
state_target_map = train_data.groupby('State')['Yield'].mean().to_dict()
dist_target_map = train_data.groupby('Dist Code')['Yield'].mean().to_dict()
global_mean = train_data['Yield'].mean()

# Apply to Train
train_data['State_Encoded'] = train_data['State'].map(state_target_map).fillna(global_mean)
train_data['Dist_Encoded'] = train_data['Dist Code'].map(dist_target_map).fillna(global_mean)

# Apply to Test
test_data['State_Encoded'] = test_data['State'].map(state_target_map).fillna(global_mean)
test_data['Dist_Encoded'] = test_data['Dist Code'].map(dist_target_map).fillna(global_mean)

# Save the target encoding dictionaries for the Streamlit Frontend!
with open('india_state_target_map.pkl', 'wb') as f:
    pickle.dump(state_target_map, f)
with open('india_dist_target_map.pkl', 'wb') as f:
    pickle.dump(dist_target_map, f)

print(f"Training set (1995-2013): {len(train_data)} rows")
print(f"Testing set (2014-2017): {len(test_data)} rows")

Training set (1995-2013): 3306 rows
Testing set (2014-2017): 696 rows


### 3: Model Training
#### Part 1: Linear Regression, Random Forest, Gradient Boosting, XGBoost

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# 1. Define Features (X) and Target (y)
features = ['State_Encoded', 'Dist_Encoded', 'Year', 'Yield_Lag1', 'Avg_Season_Tmax',
            'Total_Season_Rain', 'Sowing_Rainfall', 'Terminal_Heat_Tmax',
            'Temperature_Anomaly', 'Rainfall_Deviation']

X_train = train_data[features]
y_train = train_data['Yield']

X_test = test_data[features]
y_test = test_data['Yield']

# 2. Initialize Models
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, random_state=42, objective='reg:squarederror')
}

# 3. Train and Evaluate
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    r2 = r2_score(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    mae = mean_absolute_error(y_test, predictions)

    results.append({"Model": name, "R2 Score": r2, "RMSE": rmse, "MAE": mae})

# 4. Print Leaderboard
leaderboard = pd.DataFrame(results).sort_values(by="R2 Score", ascending=False).reset_index(drop=True)
print("\n--- MODEL LEADERBOARD ---")
display(leaderboard)


--- MODEL LEADERBOARD ---


,Model,R2 Score,RMSE,MAE
0,Random Forest,0.640494,606.204191,459.298222
1,Linear Regression,0.635686,610.244511,463.332896
2,Gradient Boosting,0.629216,615.639605,471.459703
3,XGBoost,0.569765,663.160580,519.717089


In [ ]:
from sklearn.ensemble import VotingRegressor
from xgboost import XGBRegressor
import lightgbm as lgb
from catboost import CatBoostRegressor
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# 1. Advanced Feature Engineering: Interaction Features
# Plants react exponentially to combined heat and water stress.
train_data['Heat_Water_Stress'] = train_data['Terminal_Heat_Tmax'] / (train_data['Total_Season_Rain'] + 1)
test_data['Heat_Water_Stress'] = test_data['Terminal_Heat_Tmax'] / (test_data['Total_Season_Rain'] + 1)

train_data['Climate_Extremes'] = train_data['Temperature_Anomaly'] * train_data['Rainfall_Deviation']
test_data['Climate_Extremes'] = test_data['Temperature_Anomaly'] * test_data['Rainfall_Deviation']

# Update Features List
advanced_features = features + ['Heat_Water_Stress', 'Climate_Extremes']

X_train_adv = train_data[advanced_features]
X_test_adv = test_data[advanced_features]

# 2. Initialize Models with deeper trees and slower learning rates for max precision
xgb_adv = XGBRegressor(n_estimators=800, learning_rate=0.03, max_depth=8, subsample=0.8, colsample_bytree=0.8, random_state=42, objective='reg:squarederror')
lgb_adv = lgb.LGBMRegressor(n_estimators=800, learning_rate=0.03, max_depth=8, subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1)
cat_adv = CatBoostRegressor(iterations=800, learning_rate=0.03, depth=8, random_seed=42, verbose=0)

# 3. Voting Ensemble (Average predictions of the 3 heavyweight models)
# We give a slightly higher weight to CatBoost as it often handles categorical/target-encoded data best
voting_model = VotingRegressor(estimators=[
    ('xgb', xgb_adv),
    ('lgb', lgb_adv),
    ('cat', cat_adv)
], weights=[1, 1, 1.2])

# 4. Train and Evaluate
voting_model.fit(X_train_adv, y_train)
voting_preds = voting_model.predict(X_test_adv)

vote_r2 = r2_score(y_test, voting_preds)
vote_rmse = np.sqrt(mean_squared_error(y_test, voting_preds))
vote_mae = mean_absolute_error(y_test, voting_preds)

# 5. Update Leaderboard
new_result = pd.DataFrame([{"Model": "Advanced Voting Ensemble (+ Interaction Features)", "R2 Score": vote_r2, "RMSE": vote_rmse, "MAE": vote_mae}])
leaderboard = pd.concat([leaderboard, new_result], ignore_index=True).sort_values(by="R2 Score", ascending=False).reset_index(drop=True)

print("\n--- UPDATED MODEL LEADERBOARD ---")
display(leaderboard)


--- UPDATED MODEL LEADERBOARD ---


,Model,R2 Score,RMSE,MAE
0,Random Forest,0.640494,606.204191,459.298222
1,Linear Regression,0.635686,610.244511,463.332896
2,Advanced Voting Ensemble (+ Interaction Features),0.629487,615.414349,469.899456
3,Gradient Boosting,0.629216,615.639605,471.459703
4,XGBoost,0.569765,663.160580,519.717089


In [ ]:
from sklearn.ensemble import VotingRegressor
from xgboost import XGBRegressor
import lightgbm as lgb
from catboost import CatBoostRegressor
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# 1. Advanced Feature Engineering: Interaction Features
# Plants react exponentially to combined heat and water stress.
train_data['Heat_Water_Stress'] = train_data['Terminal_Heat_Tmax'] / (train_data['Total_Season_Rain'] + 1)
test_data['Heat_Water_Stress'] = test_data['Terminal_Heat_Tmax'] / (test_data['Total_Season_Rain'] + 1)

train_data['Climate_Extremes'] = train_data['Temperature_Anomaly'] * train_data['Rainfall_Deviation']
test_data['Climate_Extremes'] = test_data['Temperature_Anomaly'] * test_data['Rainfall_Deviation']

# Update Features List
advanced_features = features + ['Heat_Water_Stress', 'Climate_Extremes']

X_train_adv = train_data[advanced_features]
X_test_adv = test_data[advanced_features]

# 2. Initialize Models with deeper trees and slower learning rates for max precision
xgb_adv = XGBRegressor(n_estimators=800, learning_rate=0.03, max_depth=8, subsample=0.8, colsample_bytree=0.8, random_state=42, objective='reg:squarederror')
lgb_adv = lgb.LGBMRegressor(n_estimators=800, learning_rate=0.03, max_depth=8, subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1)
cat_adv = CatBoostRegressor(iterations=800, learning_rate=0.03, depth=8, random_seed=42, verbose=0)

# 3. Voting Ensemble (Average predictions of the 3 heavyweight models)
# We give a slightly higher weight to CatBoost as it often handles categorical/target-encoded data best
voting_model = VotingRegressor(estimators=[
    ('xgb', xgb_adv),
    ('lgb', lgb_adv),
    ('cat', cat_adv)
], weights=[1, 1, 1.2])

# 4. Train and Evaluate
voting_model.fit(X_train_adv, y_train)
voting_preds = voting_model.predict(X_test_adv)

vote_r2 = r2_score(y_test, voting_preds)
vote_rmse = np.sqrt(mean_squared_error(y_test, voting_preds))
vote_mae = mean_absolute_error(y_test, voting_preds)

# 5. Update Leaderboard
new_result = pd.DataFrame([{"Model": "Advanced Voting Ensemble (+ Interaction Features)", "R2 Score": vote_r2, "RMSE": vote_rmse, "MAE": vote_mae}])
leaderboard = pd.concat([leaderboard, new_result], ignore_index=True).sort_values(by="R2 Score", ascending=False).reset_index(drop=True)

print("\n--- UPDATED MODEL LEADERBOARD ---")
display(leaderboard)


--- UPDATED MODEL LEADERBOARD ---


,Model,R2 Score,RMSE,MAE
0,Random Forest,0.640494,606.204191,459.298222
1,Linear Regression,0.635686,610.244511,463.332896
2,Advanced Voting Ensemble (+ Interaction Features),0.629487,615.414349,469.899456
3,Advanced Voting Ensemble (+ Interaction Features),0.629487,615.414349,469.899456
4,Gradient Boosting,0.629216,615.639605,471.459703
5,XGBoost,0.569765,663.160580,519.717089


### 4: Model Refinement
#### Systematic Hyperparameter Tuning for LightGBM (with TimeSeriesSplit)

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# 1. Prepare features (using the advanced features we created earlier)
X_train_tune = train_data[advanced_features]
y_train_tune = train_data['Yield']

X_test_tune = test_data[advanced_features]
y_test_tune = test_data['Yield']

# 2. Define the base model
lgb_base = lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)

# 3. Define the Hyperparameter Search Space
param_dist = {
    'n_estimators': [300, 500, 800, 1000],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'max_depth': [4, 6, 8, 10],
    'num_leaves': [15, 31, 63, 127],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_samples': [10, 20, 50]
}

# 4. Use TimeSeriesSplit for Cross-Validation to prevent future-data leakage
tscv = TimeSeriesSplit(n_splits=5)

# 5. Initialize RandomizedSearchCV
# n_iter=20 means it will test 20 different random combinations of the parameters above
random_search = RandomizedSearchCV(
    estimator=lgb_base,
    param_distributions=param_dist,
    n_iter=20,
    scoring='r2',
    cv=tscv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# 6. Execute the Search (This may take a minute)
print("Starting Hyperparameter Tuning...")
random_search.fit(X_train_tune, y_train_tune)

print("\nBest Parameters Found:")
print(random_search.best_params_)

# 7. Evaluate the Tuned Model on the unseen Test Set
best_lgb = random_search.best_estimator_
tuned_preds = best_lgb.predict(X_test_tune)

tuned_r2 = r2_score(y_test_tune, tuned_preds)
tuned_rmse = np.sqrt(mean_squared_error(y_test_tune, tuned_preds))
tuned_mae = mean_absolute_error(y_test_tune, tuned_preds)

# 8. Add to Leaderboard
new_result = pd.DataFrame([{"Model": "LightGBM (Tuned)", "R2 Score": tuned_r2, "RMSE": tuned_rmse, "MAE": tuned_mae}])
leaderboard = pd.concat([leaderboard, new_result], ignore_index=True).sort_values(by="R2 Score", ascending=False).reset_index(drop=True)

print("\n--- FINAL OPTIMIZED LEADERBOARD ---")
display(leaderboard)

Starting Hyperparameter Tuning...
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best Parameters Found:
{'subsample': 0.8, 'num_leaves': 15, 'n_estimators': 300, 'min_child_samples': 20, 'max_depth': 6, 'learning_rate': 0.03, 'colsample_bytree': 0.6}

--- FINAL OPTIMIZED LEADERBOARD ---


,Model,R2 Score,RMSE,MAE
0,LightGBM (Tuned),0.670273,580.554448,439.706266
1,Random Forest,0.640494,606.204191,459.298222
2,Linear Regression,0.635686,610.244511,463.332896
3,Advanced Voting Ensemble (+ Interaction Features),0.629487,615.414349,469.899456
4,Advanced Voting Ensemble (+ Interaction Features),0.629487,615.414349,469.899456
5,Gradient Boosting,0.629216,615.639605,471.459703
6,XGBoost,0.569765,663.160580,519.717089


In [ ]:
# =========================
# NEW IMPROVED MODEL CODE
# =========================

# pip install lightgbm scikit-learn pandas numpy

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import pickle

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from lightgbm import LGBMRegressor

# -------------------------
# 1. LOAD DATA
# -------------------------
file_path = "/content/drive/MyDrive/wheat_yield_india/dataset/final_india_wheat_dataset.csv"   # change path if needed
df = pd.read_csv(file_path)

# Keep useful years only
df = df[(df["Year"] >= 1995) & (df["Year"] <= 2017)].copy()

# Sort properly
df = df.sort_values(["State", "Dist Code", "Year"]).reset_index(drop=True)

print("Dataset shape:", df.shape)
print("Year range:", df["Year"].min(), "to", df["Year"].max())

# -------------------------
# 2. TRAIN / TEST SPLIT
# -------------------------
# Train: 1995-2013
# Test : 2014-2017
train_df = df[df["Year"] <= 2013].copy()
test_df  = df[df["Year"] > 2013].copy()

print("Train rows:", len(train_df))
print("Test rows :", len(test_df))

# -------------------------
# 3. SAFE TARGET ENCODING
# -------------------------
# Smoothed encoding is better than plain mean encoding
def smoothed_target_encoding(train, test, col, target="Yield", alpha=20):
    global_mean = train[target].mean()
    stats = train.groupby(col)[target].agg(["mean", "count"]).reset_index()
    stats["encoded"] = (
        (stats["count"] * stats["mean"] + alpha * global_mean)
        / (stats["count"] + alpha)
    )
    mapping = dict(zip(stats[col], stats["encoded"]))

    train_encoded = train[col].map(mapping).fillna(global_mean)
    test_encoded = test[col].map(mapping).fillna(global_mean)

    return train_encoded, test_encoded, mapping

train_df["State_Enc"], test_df["State_Enc"], state_map = smoothed_target_encoding(
    train_df, test_df, col="State", alpha=15
)

train_df["District_Enc"], test_df["District_Enc"], dist_map = smoothed_target_encoding(
    train_df, test_df, col="Dist Code", alpha=25
)

# Save encoders
with open("india_state_target_map.pkl", "wb") as f:
    pickle.dump(state_map, f)

with open("india_dist_target_map.pkl", "wb") as f:
    pickle.dump(dist_map, f)

# -------------------------
# 4. FEATURE ENGINEERING
# -------------------------
def add_features(data):
    data = data.copy()

    eps = 1e-6

    # Time feature
    data["Year_Index"] = data["Year"] - data["Year"].min()

    # Interactions
    data["Heat_Water_Stress"] = data["Terminal_Heat_Tmax"] / (data["Total_Season_Rain"] + 1.0)
    data["Climate_Extremes"] = data["Temperature_Anomaly"] * data["Rainfall_Deviation"]
    data["Lag_Heat_Interaction"] = data["Yield_Lag1"] * data["Terminal_Heat_Tmax"]
    data["Lag_Rain_Interaction"] = data["Yield_Lag1"] * data["Total_Season_Rain"]

    # Nonlinear terms
    data["Temp_Anomaly_Sq"] = data["Temperature_Anomaly"] ** 2
    data["Rain_Deviation_Sq"] = data["Rainfall_Deviation"] ** 2
    data["Terminal_Heat_Sq"] = data["Terminal_Heat_Tmax"] ** 2

    # Relative climate features
    data["Terminal_vs_AvgTemp"] = data["Terminal_Heat_Tmax"] - data["Avg_Season_Tmax"]
    data["SowingRain_to_TotalRain"] = data["Sowing_Rainfall"] / (data["Total_Season_Rain"] + 1.0)
    data["Rain_per_Temp"] = data["Total_Season_Rain"] / (data["Avg_Season_Tmax"] + eps)

    return data

train_df = add_features(train_df)
test_df = add_features(test_df)

feature_cols = [
    "State_Enc",
    "District_Enc",
    "Year",
    "Year_Index",
    "Yield_Lag1",
    "Avg_Season_Tmax",
    "Total_Season_Rain",
    "Sowing_Rainfall",
    "Terminal_Heat_Tmax",
    "Temperature_Anomaly",
    "Rainfall_Deviation",
    "Heat_Water_Stress",
    "Climate_Extremes",
    "Lag_Heat_Interaction",
    "Lag_Rain_Interaction",
    "Temp_Anomaly_Sq",
    "Rain_Deviation_Sq",
    "Terminal_Heat_Sq",
    "Terminal_vs_AvgTemp",
    "SowingRain_to_TotalRain",
    "Rain_per_Temp",
]

target_col = "Yield"

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

# -------------------------
# 5. TIME-SERIES CV SETUP
# -------------------------
# Very important: use time-aware validation, not random split
tscv = TimeSeriesSplit(n_splits=4)

# -------------------------
# 6. MODEL + HYPERPARAMETER SEARCH
# -------------------------
base_model = LGBMRegressor(
    objective="regression",
    random_state=42,
    n_jobs=-1
)

param_dist = {
    "n_estimators": [200, 400, 600, 800, 1000, 1200],
    "learning_rate": [0.01, 0.02, 0.03, 0.05, 0.08],
    "num_leaves": [15, 31, 50, 70, 100],
    "max_depth": [-1, 4, 6, 8, 10, 12],
    "min_child_samples": [5, 10, 20, 30, 40, 60],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "reg_alpha": [0.0, 0.1, 0.5, 1.0, 2.0],
    "reg_lambda": [0.0, 0.1, 0.5, 1.0, 2.0, 5.0]
}

search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dist,
    n_iter=40,
    scoring="r2",
    cv=tscv,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)

best_model = search.best_estimator_

print("\nBest Params:")
print(search.best_params_)
print("Best CV R2:", search.best_score_)

# -------------------------
# 7. FINAL TRAIN + TEST EVALUATION
# -------------------------
best_model.fit(X_train, y_train)
preds = best_model.predict(X_test)

r2 = r2_score(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
mae = mean_absolute_error(y_test, preds)

print("\n=== FINAL TEST RESULTS ===")
print(f"R2   : {r2:.4f}")
print(f"RMSE : {rmse:.2f}")
print(f"MAE  : {mae:.2f}")

# -------------------------
# 8. FEATURE IMPORTANCE
# -------------------------
feat_imp = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": best_model.feature_importances_
}).sort_values("Importance", ascending=False)

print("\nTop 15 Feature Importances:")
print(feat_imp.head(15).to_string(index=False))

# -------------------------
# 9. SAVE MODEL
# -------------------------
with open("best_lgbm_india_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

print("\nSaved:")
print("- best_lgbm_india_model.pkl")
print("- india_state_target_map.pkl")
print("- india_dist_target_map.pkl")

Dataset shape: (4002, 13)
Year range: 1995 to 2017
Train rows: 3306
Test rows : 696
Fitting 4 folds for each of 40 candidates, totalling 160 fits

Best Params:
{'subsample': 1.0, 'reg_lambda': 0.1, 'reg_alpha': 0.5, 'num_leaves': 15, 'n_estimators': 1000, 'min_child_samples': 20, 'max_depth': 8, 'learning_rate': 0.01, 'colsample_bytree': 0.8}
Best CV R2: 0.8055755351755208

=== FINAL TEST RESULTS ===
R2   : 0.6650
RMSE : 585.18
MAE  : 452.58

Top 15 Feature Importances:
                Feature  Importance
           District_Enc        2555
             Yield_Lag1        1444
                   Year        1114
   Lag_Heat_Interaction        1096
   Lag_Rain_Interaction        1093
    Terminal_vs_AvgTemp         865
        Avg_Season_Tmax         644
       Climate_Extremes         614
     Rainfall_Deviation         569
    Temperature_Anomaly         564
        Temp_Anomaly_Sq         534
      Rain_Deviation_Sq         453
SowingRain_to_TotalRain         451
        Sowing_Rainfa

In [ ]:
# pip install catboost scikit-learn pandas numpy xgboost

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import pickle

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# -----------------------------
# 1. Load Dataset & Split
# -----------------------------
df = pd.read_csv('/content/drive/MyDrive/wheat_yield_india/dataset/final_india_wheat_dataset_moderate_v2.csv')

train_df = df[df['Year'] <= 2013].copy()
test_df  = df[df['Year'] > 2013].copy()

target = 'Yield'
cat_features = ['State', 'Dist Code', 'District']
feature_cols = [c for c in df.columns if c not in [target, 'Country']]

y_train = train_df[target]
y_test  = test_df[target]

# -----------------------------
# 2. Prepare Data Formats
# -----------------------------
# Format A: One-Hot Encoded (For Scikit-Learn Models)
X_train_ohe = pd.get_dummies(train_df[feature_cols], columns=cat_features, drop_first=True)
X_test_ohe  = pd.get_dummies(test_df[feature_cols], columns=cat_features, drop_first=True)
X_train_ohe, X_test_ohe = X_train_ohe.align(X_test_ohe, join='left', axis=1, fill_value=0)

# Format B: Native Categorical (For XGBoost & CatBoost)
for col in cat_features:
    train_df[col] = train_df[col].astype('category')
    test_df[col]  = test_df[col].astype('category')

X_train_cat = train_df[feature_cols]
X_test_cat  = test_df[feature_cols]

# Array to store results for the leaderboard
results = []

def evaluate_model(name, model, X_train, X_test):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    results.append({"Model": name, "R2 Score": r2, "RMSE": rmse, "MAE": mae})
    return model

# ======================================================
# 3. Train Scikit-Learn Models (Using OHE Data)
# ======================================================

print("Training Linear Models...")
evaluate_model("Linear Regression", LinearRegression(), X_train_ohe, X_test_ohe)
evaluate_model("Ridge Regression", Ridge(alpha=1.0), X_train_ohe, X_test_ohe)

print("Training Scikit-Learn Tree Models (Using OHE)...")
evaluate_model("Random Forest", RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1), X_train_ohe, X_test_ohe)
evaluate_model("Gradient Boosting", GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42), X_train_ohe, X_test_ohe)

# ======================================================
# 4. Train Advanced Models (Using Native Categories)
# ======================================================

print("Training XGBoost (Native Categories)...")
xgb_model = XGBRegressor(
    n_estimators=300, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    tree_method='hist', enable_categorical=True
)
evaluate_model("XGBoost", xgb_model, X_train_cat, X_test_cat)

print("Training CatBoost (Native Categories)...")
cat_model = CatBoostRegressor(
    iterations=300, depth=6, learning_rate=0.1,
    loss_function='RMSE', random_seed=42, logging_level='Silent'
)
# CatBoost requires specific categorical feature indexing
cat_model.fit(X_train_cat, y_train, cat_features=cat_features)
preds_cat = cat_model.predict(X_test_cat)
results.append({
    "Model": "CatBoost",
    "R2 Score": r2_score(y_test, preds_cat),
    "RMSE": np.sqrt(mean_squared_error(y_test, preds_cat)),
    "MAE": mean_absolute_error(y_test, preds_cat)
})

# ======================================================
# 5. Display Leaderboard
# ======================================================
leaderboard = pd.DataFrame(results).sort_values(by="R2 Score", ascending=False).reset_index(drop=True)
print("\n========================================")
print("   FINAL MODEL LEADERBOARD (WITH GEO)")
print("========================================")
display(leaderboard)



Training Linear Models...
Training Scikit-Learn Tree Models (Using OHE)...
Training XGBoost (Native Categories)...
Training CatBoost (Native Categories)...

   FINAL MODEL LEADERBOARD (WITH GEO)


,Model,R2 Score,RMSE,MAE
0,Ridge Regression,0.894889,327.785041,243.527393
1,Linear Regression,0.894870,327.815130,243.624562
2,Gradient Boosting,0.878772,352.020287,260.750147
3,Random Forest,0.871622,362.252256,263.797964
4,CatBoost,0.870696,363.556532,269.434671
5,XGBoost,0.823723,424.485990,328.353700


In [ ]:
import pickle
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor

# 1. Train the Gradient Boosting model on the One-Hot Encoded data
gb_model_india = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

# Note: X_train_ohe and y_train must already be in your notebook's memory
# from the previous benchmarking script.
gb_model_india.fit(X_train_ohe, y_train)

# 2. Define Save Paths
model_save_path = '/content/drive/MyDrive/wheat_yield_india/dataset/india_gb_model.pkl'
metadata_save_path = '/content/drive/MyDrive/wheat_yield_india/dataset/india_gb_metadata.pkl'

# 3. Save the Model
with open(model_save_path, 'wb') as f:
    pickle.dump(gb_model_india, f)

# 4. Save the Metadata (CRITICAL: Must save the OHE columns, not the raw features)
ohe_features = list(X_train_ohe.columns)

metadata = {
    'features': ohe_features, # The frontend will use this to align the columns
    'cat_features': cat_features,
    'target': target
}

with open(metadata_save_path, 'wb') as f:
    pickle.dump(metadata, f)

print("India Gradient Boosting model and OHE metadata exported successfully!")

India Gradient Boosting model and OHE metadata exported successfully!
